**`US_curate_footprints`**

This pipeline was developed to build a footprint-level building inventory
for hurricane damage modeling in the United States.

It runs the complete recipe chain: ingest, harmonize, enrich, and curate.

Parcels are curated first (`US_parcel-openplaces-2026`) to get a land-use classification for use in footprint curation.

Pass `--no_streetview` to skip Google Street View  (ingest and `n_stories` enrichment) to avoid its
per-request billing while iterating.

Pass `--no_googlesatellite` to skip Google Satellite (ingest and roof-shape enrichment) to avoid its
per-request billing while iterating.

Recipe: `US_footprint-cheer-2026.yaml`

File: `src/openplaces/recipes/US/_all/footprint/cheer/2026/US_footprint-cheer-2026.yaml`

# Configure

In [ ]:
import argparse

from openplaces.core.schema import AdminId
from openplaces.geo.link import create_entity_link
from openplaces.io.cleanup import cleanup
from openplaces.io.curator import curate
from openplaces.io.enricher import enrich
from openplaces.io.harmonizer import harmonize
from openplaces.io.ingester import ingest
from openplaces.io.ingester.image_ingester import ImageScraperError
from openplaces.recipe import find_entity_recipe_id, get_output_path, get_recipe_by_id
from openplaces.timing import get_timer

In [ ]:
parser = argparse.ArgumentParser(
    description='Ingest, harmonize, enrich, and curate footprints'
)
parser.add_argument(
    '--recipe_id',
    help='Curation recipe (e.g. "US_footprint-cheer-2026")',
)
parser.add_argument(
    '--admin_ids',
    help='Admin unit IDs to process (e.g. "US-NC-BS")',
    nargs='*',
)
parser.add_argument(
    '--reprocess',
    help='Reprocess admin IDs even if output already exists',
    action='store_true',
)
parser.add_argument(
    '--redownload',
    help='Redownload input data from original source',
    action='store_true',
)
parser.add_argument(
    '--keep_unzipped',
    help='Keep unzipped datasets in heap folder after processing',
    action='store_true',
)
parser.add_argument(
    '--no_streetview',
    help=(
        'Skip Google Street View entirely (ingest and the n_stories '
        'enrichment step that depends on it), e.g. to avoid its '
        'per-request billing while iterating'
    ),
    action='store_true',
)
parser.add_argument(
    '--no_googlesatellite',
    help=(
        'Skip Google Satellite entirely (ingest and the roof-shape '
        'enrichment step that depends on it), e.g. to avoid its '
        'per-request billing while iterating'
    ),
    action='store_true',
)
parser.add_argument(
    '--cleanup',
    help=(
        'Reclaim consumed intermediate files as stages finish: "consumed" '
        'deletes cache parquets (and image caches when opted in via '
        'retention.cleanup.include_images) once all their consumers are '
        'complete; "aggressive" additionally treats core spines as '
        'reclaimable once the curated outputs exist'
    ),
    default='none',
    choices=['none', 'consumed', 'aggressive'],
)
parser.add_argument(
    '--verbose',
    action='store_true',
)

# Test arguments

In [ ]:
ARGS_TEST = (
    '--recipe_id US_footprint-cheer-2026 '
    '--admin_ids US-NC-CE '  # Carteret (pilot)
    # '--admin_ids US-NC-CE-MO '  # Morehead township
    # '--admin_ids US-NC-BS '  # Brunswick
    # '--admin_ids US-NC-BS-SH '  # Shallotte township
    # '--admin_ids US-MA-MI-SO '  # Somerville
    # '--admin_ids US-MA-SU '  # Suffolk county (Boston)
    # '--admin_ids US-FL-AL '  # Alachua
    # '--admin_ids US-TX-JE '
    '--reprocess '
    # '--redownload '
    # '--keep_unzipped '
    '--no_googlesatellite '
    '--no_streetview '
    # '--cleanup consumed '
    '--verbose '
)

args_list = [x for x in ARGS_TEST.split(' ') if x]
args = parser.parse_args(args_list)
args

In [ ]:
# Pretty-print recipes
from openplaces.utils import pretty_print

curation_recipe = get_recipe_by_id(args.recipe_id)
harmonization_recipe_id = curation_recipe['entity_recipe']
enrichment_recipe_ids = [
    spec['recipe_id']
    for step in curation_recipe['pipeline']
    for spec in step.get('recipes', [])
]
# Parcel curation lane consumed by the footprint recipe's link_curated_entity
# step. Parcels are curated first; the footprint recipe joins their land-use
# (e.g. the manufactured_home_park flag) by parcel_id_local.
parcel_curation_recipe_id = next(
    (
        step['recipe_id']
        for step in curation_recipe['pipeline']
        if step['step'] == 'link_curated_entity'
    ),
    None,
)
parcel_harmonization_recipe_id = (
    get_recipe_by_id(parcel_curation_recipe_id)['entity_recipe']
    if parcel_curation_recipe_id
    else None
)

print('Footprint curation recipe (produces final output):\n')
pretty_print(curation_recipe)

print('\nPrecursor recipes:\n\nFootprint harmonization recipe:\n')
pretty_print(get_recipe_by_id(harmonization_recipe_id))
if parcel_harmonization_recipe_id:
    print('\nParcel harmonization recipe:')
    pretty_print(get_recipe_by_id(parcel_harmonization_recipe_id))
for enrichment_recipe_id in enrichment_recipe_ids:
    print('\nEnrichment recipe:')
    pretty_print(get_recipe_by_id(enrichment_recipe_id))
if parcel_curation_recipe_id:
    print('\nParcel curation recipe:')
    pretty_print(get_recipe_by_id(parcel_curation_recipe_id))

# Ingest precursor datasets

In [ ]:
curation_recipe = get_recipe_by_id(args.recipe_id)
harmonization_recipe_id = curation_recipe['entity_recipe']
enrichment_recipe_ids = [
    spec['recipe_id']
    for step in curation_recipe['pipeline']
    for spec in step.get('recipes', [])
]
# Parcel curation lane (curated before footprints; see link_curated_entity).
parcel_curation_recipe_id = next(
    (
        step['recipe_id']
        for step in curation_recipe['pipeline']
        if step['step'] == 'link_curated_entity'
    ),
    None,
)
parcel_harmonization_recipe_id = (
    get_recipe_by_id(parcel_curation_recipe_id)['entity_recipe']
    if parcel_curation_recipe_id
    else None
)

# Save keyword arguments that will be passed to all ingest functions
ingest_kwargs = {
    key: getattr(args, key)
    for key in [
        'reprocess',
        'redownload',
        'keep_unzipped',
        'verbose',
    ]
}

# Harmonize and curate run at the recipes' process level (admin level 3).
# Truncate finer-grained admin IDs (e.g. a township) to their county;
# ingest and enrich calls below take args.admin_ids directly: ingest
# self-truncates (keeping the requested level for image recipes), and
# enrich restricts image-based steps to the requested units.
process_admin_ids = list(
    dict.fromkeys(str(AdminId(*AdminId(a).levels[:3])) for a in args.admin_ids)
)

# Track stage runtimes; saved to the logs directory at the end of the run
timer = get_timer(
    'US_curate_footprints',
    admin_id=process_admin_ids[0] if len(process_admin_ids) == 1 else None,
    verbose=args.verbose,
    overwrite=True,
    recipe_id=args.recipe_id,
    admin_ids=args.admin_ids,
)

## Admin boundaries

In [ ]:
# US admin boundaries (for allocating Microsoft footprints to counties)
ingest('US_admin-census-2021_admin2', **ingest_kwargs)
timer.mark('ingest US_admin-census-2021_admin2')

ingest('US_admin-census-2021_admin3', **ingest_kwargs)
timer.mark('ingest US_admin-census-2021_admin3')

# Image recipes fetch at admin level 4 (townships)
ingest('US_admin-census-2021_admin4', **ingest_kwargs)
timer.mark('ingest US_admin-census-2021_admin4')

## Tiles

In [ ]:
# Tile-partitioned recipes (e.g. footprint-obm-2025) resolve which tiles to
# download for an admin unit via a precomputed tile<->admin overlay link
# (see notebooks/02_ingest/tiles/ingest_tiles.ipynb). That link is not built
# automatically by ingest(), so build it here if missing.
obm_recipe = get_recipe_by_id('footprint-obm-2025')
tile_recipe_id = obm_recipe['download_by']['tile_recipe_id']
admin_recipe_id = obm_recipe['overlay_admin_ids']['admin_recipe_id']

ingest(tile_recipe_id, **ingest_kwargs)
timer.mark(f'ingest {tile_recipe_id}')

tiles_path = get_output_path(get_recipe_by_id(tile_recipe_id))
tile_admin_link_path = tiles_path.with_name(
    tiles_path.stem + f'_{admin_recipe_id}.parquet'
)
if not tile_admin_link_path.exists():
    create_entity_link(tile_recipe_id, admin_recipe_id)
    timer.mark(f'link {tile_recipe_id} to {admin_recipe_id}')

## Footprints

In [ ]:
# Global building footprints: OpenBuildingsMap (OBM)
ingest('footprint-obm-2025', admin_ids=args.admin_ids, **ingest_kwargs)
timer.mark('ingest footprint-obm-2025')

In [ ]:
# US building footprints: Microsoft
ingest('US_footprint-microsoft-v2', admin_ids=args.admin_ids, **ingest_kwargs)
timer.mark('ingest US_footprint-microsoft-v2')

In [ ]:
# State footprints: auto-discovered per state from args.admin_ids
state_groups = {}
for aid_str in args.admin_ids or []:
    aid = AdminId(aid_str)
    if aid.get_level() >= 2:
        state_id = str(AdminId(*aid.levels[:2]))
        state_groups.setdefault(state_id, []).append(aid_str)

for state_id, children in state_groups.items():
    recipe_id = find_entity_recipe_id(
        state_id, 'footprint', stage='ingest', silent=True
    )
    if recipe_id:
        ingest(recipe_id, admin_ids=children, **ingest_kwargs)
timer.mark('ingest state footprints')

In [ ]:
# US building footprints: FEMA USA Structures.
# Ingested for parcel-occupancy linkage, not as a footprint-spine geometry source
# (FEMA caused IoU-merge errors). The parcel spine attributes FEMA's dominant
# occupancy per parcel; that flows to footprints as occupancy evidence.
ingest('US_footprint-fema-2023', admin_ids=args.admin_ids, **ingest_kwargs)
timer.mark('ingest US_footprint-fema-2023')

## Parcels

In [ ]:
# State parcels: auto-discovered per state from args.admin_ids
for state_id, children in state_groups.items():
    recipe_id = find_entity_recipe_id(state_id, 'parcel', stage='ingest', silent=True)
    if recipe_id:
        ingest(recipe_id, admin_ids=children, **ingest_kwargs)
timer.mark('ingest state parcels')

## Buildings

In [ ]:
# US building points: National Structure Inventory (NSI)
ingest('US_building-nsi-2022', admin_ids=args.admin_ids, **ingest_kwargs)
timer.mark('ingest US_building-nsi-2022')

## Dwellings (address points)

In [ ]:
# Global dwelling points: Overture
ingest('dwelling-overture-2025', admin_ids=args.admin_ids, **ingest_kwargs)
timer.mark('ingest dwelling-overture-2025')

# Harmonize

Harmonize the footprint spine, then the parcel spine (which reads the footprint
spine to summarize per-parcel footprint morphology).

## Harmonize footprints

In [ ]:
harmonize(
    harmonization_recipe_id,
    admin_ids=process_admin_ids,
    reprocess=args.reprocess,
    verbose=args.verbose,
)
timer.mark('harmonize')

## Harmonize parcels

In [ ]:
# Harmonize the parcel spine: links parcels to local tax records and the NSI
# building group, and attaches per-parcel footprint morphology (reads the
# footprint spine above). This is the input to the parcel curation lane.
if parcel_harmonization_recipe_id:
    harmonize(
        parcel_harmonization_recipe_id,
        admin_ids=process_admin_ids,
        reprocess=args.reprocess,
        verbose=args.verbose,
    )
    timer.mark('harmonize parcels')

# Both spines exist now, so the ingested cache parquets they consumed
# (footprints, parcels, NSI, Overture) have no incomplete consumers left;
# reclaim them (each deletion leaves a tombstone receipt so a rerun still
# skips the ingest).
if args.cleanup != 'none':
    cleanup(
        args.recipe_id,
        admin_ids=process_admin_ids,
        stages=('ingest',),
        dry_run=False,
        verbose=args.verbose,
    )
    timer.mark('cleanup ingested inputs')

# Enrich

## Ingest images

In [ ]:
image_recipe_ids = [
    get_recipe_by_id(recipe_id).get('image_recipe')
    for recipe_id in enrichment_recipe_ids
]

for image_recipe_id in dict.fromkeys(image_recipe_ids):
    if image_recipe_id is None:
        continue
    if args.no_streetview and 'streetview' in image_recipe_id:
        print(f'[skip] {image_recipe_id}: --no_streetview')
        continue
    if args.no_googlesatellite and 'googlesatellite' in image_recipe_id:
        print(f'[skip] {image_recipe_id}: --no_googlesatellite')
        continue
    try:
        ingest(image_recipe_id, admin_ids=args.admin_ids, **ingest_kwargs)
    except ImageScraperError as exception:
        # A scraper that can't be initialized (e.g. Street View when its Google
        # Cloud project lacks billing) should not abort the whole batch.
        print(f'[skip] {image_recipe_id}: {exception}')
        continue
    timer.mark(f'ingest {image_recipe_id}')

## Enrich footprints with images

In [ ]:
if args.no_streetview:
    # Drop enrichment recipes that depend on Street View entirely, rather
    # than running enrich() to a no-op on imagery that was never ingested.
    enrichment_recipe_ids = [
        recipe_id
        for recipe_id in enrichment_recipe_ids
        if 'streetview' not in (get_recipe_by_id(recipe_id).get('image_recipe') or '')
    ]

if args.no_googlesatellite:
    # Drop enrichment recipes that depend on Google Satellite entirely, rather
    # than running enrich() to a no-op on imagery that was never ingested.
    enrichment_recipe_ids = [
        recipe_id
        for recipe_id in enrichment_recipe_ids
        if 'googlesatellite'
        not in (get_recipe_by_id(recipe_id).get('image_recipe') or '')
    ]

for enrichment_recipe_id in enrichment_recipe_ids:
    enrich(
        enrichment_recipe_id,
        admin_ids=args.admin_ids,
        entity_recipe_id=harmonization_recipe_id,
        # reprocess=args.reprocess,
        reprocess=False,
        verbose=args.verbose,
    )
    timer.mark(f'enrich {enrichment_recipe_id}')

# All enrichment evidence is saved; image caches are reclaimable once every
# enrich recipe sharing an image_recipe has full coverage. Deletion of the
# caches themselves stays opt-in (retention.cleanup.include_images) — the
# report always lists them.
if args.cleanup != 'none':
    cleanup(
        args.recipe_id,
        admin_ids=process_admin_ids,
        dry_run=False,
        verbose=args.verbose,
    )
    timer.mark('cleanup enrichment inputs')

# Curate

Curate parcels first (land-use classification), then footprints — the footprint
recipe consumes the curated parcel use group via `link_curated_entity`.

## Curate parcels

In [ ]:
# Curate parcels first: classify land use (manufactured-home park vs RV park vs
# other) from assessor codes, the linked-NSI group, and footprint morphology.
# The footprint curate below joins this in via link_curated_entity.
if parcel_curation_recipe_id:
    curate(
        parcel_curation_recipe_id,
        admin_ids=process_admin_ids,
        reprocess=args.reprocess,
        verbose=args.verbose,
    )
    timer.mark('curate parcels')

## Curate footprints

In [ ]:
curate(
    args.recipe_id,
    admin_ids=process_admin_ids,
    reprocess=args.reprocess,
    verbose=args.verbose,
    save_statistics=True,
)
timer.mark('curate')

# Final sweep now that the curated deliverable exists. 'aggressive'
# additionally treats the core spines as reclaimable (kept only until the
# curated outputs exist); enrichment evidence sidecars are always kept —
# their image inputs may be gone.
if args.cleanup != 'none':
    cleanup(
        args.recipe_id,
        admin_ids=process_admin_ids,
        aggressive=(args.cleanup == 'aggressive'),
        dry_run=False,
        verbose=args.verbose,
    )
    timer.mark('cleanup consumed intermediates')

In [ ]:
# Report and save stage runtimes (JSON in the logs directory)
timer.summary()
timer.save()

---
# Convert to script

*The above line and heading identify the end of the script.*

*Code below this marker will not be included in the converted `.py` script.*

In [ ]:
from openplaces.flow import convert_to_script

COMMIT = True
# If True, writes `.py` scripts to 'scripts/'.
# If False, writes a test version of the script to 'scripts/_test/'

In [ ]:
convert_to_script(commit=COMMIT)

# Test script

In [ ]:
# test_script(*args_list, committed=COMMIT)

# Loop script

In [ ]:
# CHEER_ADMIN3_IDS = 'US-NC-BA US-NC-BT US-NC-BL US-NC-BS US-NC-CD US-NC-CE US-NC-CW US-NC-CM US-NC-CN US-NC-CU US-NC-CI US-NC-DE US-NC-DP US-NC-ED US-NC-FR US-NC-GT US-NC-GE US-NC-HL US-NC-HT US-NC-HD US-NC-HO US-NC-HE US-NC-JH US-NC-JN US-NC-LN US-NC-MR US-NC-NA US-NC-NE US-NC-NO US-NC-ON US-NC-PM US-NC-PU US-NC-PD US-NC-PQ US-NC-PI US-NC-RB US-NC-SP US-NC-SC US-NC-TY US-NC-WK US-NC-WR US-NC-WI US-NC-WY US-NC-WO'.split()
# CHEER_ADMIN3_IDS = 'US-TX-AA US-TX-AU US-TX-BE US-TX-BI US-TX-BK US-TX-CU US-TX-CA US-TX-CH US-TX-CO US-TX-DE US-TX-DU US-TX-FT US-TX-FR US-TX-GV US-TX-GI US-TX-HN US-TX-RR US-TX-HG US-TX-JA US-TX-JR US-TX-JE US-TX-JG US-TX-JW US-TX-KE US-TX-KL US-TX-LA US-TX-LT US-TX-LK US-TX-MD US-TX-NE US-TX-NU US-TX-OR US-TX-RF US-TX-SP US-TX-SR US-TX-TY US-TX-VI US-TX-WR US-TX-WH US-TX-WB US-TX-WN US-TX-WY'.split()

In [ ]:
# args_list_cheer = (
#     ['--recipe_id', args.recipe_id, '--admin_ids']
#     + CHEER_ADMIN3_IDS
#     + ['--verbose', '--reprocess']
# )
# args_list_cheer

In [ ]:
# test_script(*args_list_cheer)

# Aggregate output files

In [ ]:
# from openplaces.api import aggregate_files

# aggregate_files(
#     args.recipe_id,
#     admin_level=2,
#     output_dir='share',
#     admin_ids_to_aggregate=CHEER_ADMIN3_IDS,
#     keep_original=True,
#     verbose=True,
#     combined=True,
# )

# Inspect outputs

In [ ]:
from openplaces import get_entities

footprints = get_entities(args.recipe_id, args.admin_ids, geom=True)
footprints.sample(5).T

# Inspect building imagery

For a randomly sampled building, show its footprint in context alongside the
downloaded Google Satellite and Street View images that fed the enrichment
detectors. Panels show a "not available" placeholder when no image was
downloaded for that building.

In [ ]:
from openplaces import get_entities
from openplaces.viz import show_building_imagery

# Image recipes used by this curation recipe's enrichment steps, with a
# friendly panel label per image source (satellite + street view).
LABELS = {'googlesatellite': 'Google Satellite', 'googlestreetview': 'Street View'}
image_recipes = {}
for enrichment_recipe_id in enrichment_recipe_ids:
    image_recipe_id = get_recipe_by_id(enrichment_recipe_id).get('image_recipe')
    if not image_recipe_id:
        continue
    label = next(
        (name for key, name in LABELS.items() if key in image_recipe_id),
        image_recipe_id,
    )
    image_recipes[label] = image_recipe_id

# Context map + each building's downloaded Google imagery (placeholder when an
# image was not downloaded, e.g. Street View if its ingest was skipped).
footprints = get_entities(args.recipe_id, args.admin_ids, geom=True)
building = footprints.sample(1)

show_building_imagery(
    location=building,
    geodatasets={'footprints': footprints},
    image_recipes=image_recipes,
    admin_id=args.admin_ids[0],
)

# Profile disk usage

The image caches written by this notebook can grow large (tens of GB per
county).

In [ ]:
from openplaces import diagnostics

# Disk usage by admin unit and dataset across the configured data
# directories (core, external, heap, cache, out)
usage = diagnostics.profile_disk_usage(min_size_mb=50)
usage.head(15)

In [ ]:
# Image caches by location
diagnostics.list_image_caches()

In [ ]:
# Delete with a county or township ID (dry_run=True only reports;
# pass dry_run=False to actually delete)
# from openplaces.io import delete_image_caches
# delete_image_caches(['US-NC-BS'], dry_run=True)
# delete_image_caches(args.admin_ids, dry_run=True)